<span style="color: yellow; font-weight: bold; letter-spacing: 3px;"> [sessions_conditions_2.ipynb] Созданиие расписаний торговых сессий  </span>
* загрузка данных SQL таблиц для дальнейшей работы;
* Транспонирование методом  [ pivot_table ];
* Преобразование времени из минут в формат ЧЧ:ММ;
* Формирование Таблиц и отчётов

In [1]:
# Динамический импорт и инициализация библиотек и путей для работы с данными в Python <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
from pathlib import Path
import re
import csv
import sys
import os
import pandas as pd
import json
from collections import defaultdict

file_dir = os.getcwd()                                                              # Определяем путь к текущему файлу (где выполняется код)
print(f"📂 Директория файла:                                      {file_dir}")

sub_project_dir = Path(file_dir).parent
print(f"📂 Директория СубПроекта:                     {sub_project_dir}")

project_dir                     = Path(file_dir).parent.parent.parent               # Переход в верхнюю директорию проекта (fc_to_mt5_migrations/own_platform)
print(f"📂 Рабочая директория проекта:                             {project_dir}")

parent_dir                      = Path.cwd().parent.parent.parent.parent            # Переход на уровень выше (fc_to_mt5_migrations)
print(f"📂 Рабочая директория проекта для доступа к библиотекам:   {parent_dir}")


# Функция для проверки существования директории и её создания при отсутствии <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
def ensure_directory(path, description="", name=""):
    if not os.path.isdir(path):
        os.makedirs(path)
        print(f"❗📁 [{name}] не найден, создан новый каталог: {os.path.abspath(path)}")
    else:
        print(f"📁 [{name}]; {description}: {os.path.abspath(path)}")
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

input_log_data = os.path.join(project_dir, sub_project_dir, "input_data", 'input_log_data')
ensure_directory(input_log_data, "Путь к каталогу с логами исходных файлов", "input_log_data")
print(f"📁 [input_log_data];     Путь к каталогу с историей исходных данных:{input_log_data}"); os.makedirs("tests/fixtures", exist_ok=True)

input_temp_data = os.path.join(project_dir, sub_project_dir, "input_data", 'input_temp_data')           # Путь к каталогу с входными временными файлами
print(f"📁 [input_temp_data];    Путь к каталогу с исходными данными:{input_temp_data}")
input_samples_data = os.path.join(project_dir, sub_project_dir, "input_data", 'input_samples')     # Путь к каталогу с входными примерами данных
print(f"📁 [input_samples_data]; Путь к каталогу с примерами данных: {input_samples_data}")

directory_data_log_files        = os.path.join(project_dir, sub_project_dir, 'log_data_files')
#print(f"📁 [directory_data_log_files];     Путь к каталогу с лог-файлами:                          {directory_data_log_files}")
ensure_directory(directory_data_log_files, "Путь к каталогу с лог-файлами", "directory_data_log_files")

directory_data_temp_files       = os.path.join(project_dir, sub_project_dir, 'working_data_files') 
ensure_directory(directory_data_temp_files, "Путь к каталогу с временными файлами", "directory_data_temp_files")
#print(f"📁 [directory_data_temp_files];    Путь к каталогу с временными файлами:                   {directory_data_temp_files}")

directory_data_original_data    = os.path.join(project_dir, 'original_data')        # Путь к каталогу с оригинальными данными
print(f"📁 Путь к каталогу с оригинальными данными:                {directory_data_original_data}")

directory_data_set              = os.path.join(project_dir, 'data_set')             # Путь к каталогу с конфигурационными данными
print(f"📁 Путь к каталогу с файлами настроек:                     {directory_data_set}")

directory_data_output           = os.path.join(parent_dir, 'output_data')           # Путь к каталогу с выходными данными

libraries_path = os.path.join(parent_dir, "libraries_py")                           # Формируем путь к libraries_py каталогу с библиотеками *.py
sys.path.append(libraries_path)                                                     # sys.path — это список путей, где Python ищет модули при import module_name.

if libraries_path in sys.path: print(f"✅ Success: Каталог {libraries_path} успешно добавлен в sys.path")
else: print(f"❌ ERROR: {libraries_path} не найден в sys.path")

"""sub_project_dir = "trading_conditions"

file_dir = os.getcwd()                                                              # Определяем путь к текущему файлу (где выполняется код)
print(f"Файл в директории:                                      {file_dir}")

project_dir                     = Path(file_dir).parent.parent                      # Переход на уровень выше (fc_to_mt5_migrations/own_platform)
print(f"Рабочая директория проекта:                             {project_dir}")

parent_dir                      = Path.cwd().parent.parent.parent                   # Переход на уровень выше (fc_to_mt5_migrations)
print(f"Рабочая директория проекта для доступа к библиотекам:   {parent_dir}")

directory_data_log_files        = os.path.join(project_dir, sub_project_dir, 'log_data_files')
print(f"[directory_data_log_files];     Путь к каталогу с лог-файлами:                          {directory_data_log_files}")

directory_data_temp_files       = os.path.join(project_dir, sub_project_dir, 'working_data_files') 
print(f"[directory_data_temp_files];    Путь к каталогу с временными файлами:                   {directory_data_temp_files}")

directory_data_original_data    = os.path.join(project_dir, 'original_data')        # Путь к каталогу с оригинальными данными
print(f"Путь к каталогу с оригинальными данными:                {directory_data_original_data}")

directory_data_set              = os.path.join(project_dir, 'data_set')             # Путь к каталогу с конфигурационными данными
print(f"Путь к каталогу с файлами настроек:                     {directory_data_set}")

directory_data_output           = os.path.join(parent_dir, 'output_data')           # Путь к каталогу с выходными данными

libraries_path = os.path.join(parent_dir, "libraries_py")                           # Формируем путь к libraries_py каталогу с библиотеками *.py
sys.path.append(libraries_path)                                                     # sys.path — это список путей, где Python ищет модули при import module_name.

if libraries_path in sys.path: print(f"✅ Каталог {libraries_path} успешно добавлен в sys.path")
else: print(f"❌ Ошибка: {libraries_path} не найден в sys.path")"""

# Динамически импорт необходимых функций <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
file_imports = "dynamic_import_functions.py"                                        # Библиотека для динамического импорта
file_imports_path = os.path.join(libraries_path, file_imports)
if os.path.exists(file_imports_path):
    import importlib
    importlib.invalidate_caches()                                                   # Сбрасываем кэш перед импортом
    from dynamic_import_functions import import_functions, print_import_function_info
    print(f"\n ✅ Импорт [{file_imports}] успешен.")
else:
    print(f"\n ERROR: Файл '{file_imports}' не найден по пути {file_imports_path}, импорт не выполнен.\n")

modules_to_import = {                                                               # Формируем словарь, с именами файлов и функциями в них
    "yar_sed_general_lib":
        [libraries_path,
                "pd_set_option",                     # Вывод ДФ
                "df_to_csv",                         # Сохранение ДФ в CSV 
                #"CSVLoader",
                "save_data_log_work_file",
                #"detect_encoding",
                "time_to_minutes",
                #"load_string_list",
                "list_print",
                "move_column",
                #"filter_df_by_suffix",
                "check_columns_exist_id_df",
                "merge_left_with_check"],           # Проверка наличия колонок в DataFrame 
    "sql_request_2":
        [libraries_path, 
                "pd_read_sql",
                "get_sql_tab"]
                }

imported = import_functions(modules_to_import)          # Импортируем модули из словаря modules_to_import

print_import_function_info(modules_to_import, imported) # Выводим переменные ожидаемые импортированными функциями 

📂 Директория файла:                                      c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\Reading\ipynb_files
📂 Директория СубПроекта:                     c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\Reading
📂 Рабочая директория проекта:                             c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform
📂 Рабочая директория проекта для доступа к библиотекам:   c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations
📁 [input_log_data]; Путь к каталогу с логами исходных файлов: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\Reading\input_data\input_log_data
📁 [input_log_data];     Путь к каталогу с историей исходных данных:c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\Reading\input_data\input_log_data
📁 [input_temp_data];    Путь к каталогу с исходным

In [2]:
# [ ФУНКЦИИ ] используемые в этом файле <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
print(f"ƒ [ внутренние ФУНКЦИИ ] используемые в этом файле:")    # ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''

print("ƒ[внутренняя ФУНКЦИЯ] Сортировка колонок в нужном порядке.")#<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
def sort_session_columns(cols):                                         # Функция для сортировки колонок в нужном порядке
    print("[внутренняя ФУНКЦИЯ] Сортировка колонок в нужном порядке.")
    def sort_key(col):
        if col == 'symbolId':   return (-1, '', '', '')                 # всегда первая
        try:                                                            # Разбиваем имя колонки: <type>_<field>_<day>
            parts = col.split('_')
            stype = parts[0]        # trade / quote
            field = parts[1]        # open / close
            day = int(parts[2])     # номер дня
        except: stype, field, day = '', '', 0           # для колонок, которые не подходят под шаблон
        field_order = 0 if field=='open' else 1         # Ключ сортировки: день, open/close, trade/quote
        stype_order = 0 if stype=='trade' else 1        # Ключ сортировки: день, open/close, trade/quote
        return (day, field_order, stype_order)
    return sorted(cols, key=sort_key)
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> 

# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
print("ƒ[внутренняя ФУНКЦИЯ] Проверка словарей на одинаковые ключи, значения и пары key:value.") # <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<  
def check_dicts(dicts_with_names):  
    print("ƒ[внутренняя ФУНКЦИЯ] Проверка словарей на одинаковые ключи, значения и пары key:value.")
    key_to_dicts = defaultdict(list)          # key -> [dict_name]
    value_to_entries = defaultdict(list)      # value -> [(dict_name, key)]
    kv_to_dicts = defaultdict(list)            # (key, value) -> [dict_name]

    for dict_name, d in dicts_with_names:                       # Сбор информации
        for k, v in d.items():
            key_to_dicts[k].append(dict_name)
            value_to_entries[v].append((dict_name, k))
            kv_to_dicts[(k, v)].append(dict_name)

    same_keys = {                                                   # 1. Одинаковые ключи
        k: names
        for k, names in key_to_dicts.items()
        if len(names) > 1
    }

    same_values = {                                                 # 2. Одинаковые значения
        v: entries
        for v, entries in value_to_entries.items()
        if len(entries) > 1
    }

    same_key_values = {                                             # 3. Одинаковые пары key:value
        (k, v): names
        for (k, v), names in kv_to_dicts.items()
        if len(names) > 1
    }

    return {"same_keys": same_keys,"same_values": same_values,"same_key_values": same_key_values,}
# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> [ФУНКЦИЯ] Проверка словарей на одинаковые ключи, значения и пары key:value

# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
print("ƒ[внутренняя ФУНКЦИЯ] Фильтрация DataFrame по суффиксам.")    # <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
def filter_by_suffix(df, suffix_list):
    print("ƒ[внутренняя ФУНКЦИЯ] Фильтрация DataFrame по суффиксам.")
    print(f" \n \n Создаём колонку с  сответствующими суффиксами из списка:")
    imported["list_print"](suffix_list, "suffix_list")
    df = symbols_df[symbols_df["name"].str.endswith(tuple(suffix_list))].copy()
    df["suffix"] = df["name"].str.extract(r'(\..*)')
    unique_df = df[["marketId", "suffix"]].drop_duplicates().copy()

    cols_map = {"name": "name_market"}   
    unique_df, not_found, not_used = imported['merge_left_with_check']( # Вызываем функцию слияния ДФ с проверкой наличия колонок и размерности
    df_left= unique_df, df_right= symbols_markets_df, left_on= "marketId", right_on= "id",  cols_map=cols_map)

    #imported["pd_set_option"]("unique_df", unique_df, 50)
    return unique_df

# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
print("ƒ[внутренняя ФУНКЦИЯ] Создание словаря {name_market: id} с проверкой на один ключ — несколько значений.") # <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
def create_market_dict(df, key_col='name_market', value_col='id'):          # Создаёт словарь {name_market: id} с проверкой на один ключ — несколько значений.
    print("ƒ[внутренняя ФУНКЦИЯ] Создание словаря {name_market: id} с проверкой на один ключ — несколько значений.")
    unique_counts = df.groupby(key_col)[value_col].nunique()                # Группируем по ключу и считаем количество уникальных значений
    multi_value_keys = unique_counts[unique_counts > 1].index.tolist()      # Ключи с несколькими уникальными значениями
    
    if multi_value_keys:
        print(f"Внимание: Один ключ соответствует нескольким разным значениям для {len(multi_value_keys)} ключ(ей): {multi_value_keys}")

        for key in multi_value_keys:                                        # Детали для каждого проблемного ключа
            values = df[df[key_col] == key][value_col].unique().tolist()
            print(f"  Ключ '{key}' → значения: {values}")
        
        df_dedup = df.drop_duplicates(subset=key_col, keep='first')         # Обработка: берём первое значение (или можно бросить ошибку/выбрать вручную)
        print("  Для словаря взяты первые значения для каждого ключа.")
    else:
        print("Каждому ключу соответствует ровно одно значение.")
        df_dedup = df.copy()
    
    market_dict = dict(zip(df_dedup[key_col], df_dedup[value_col]))         # Создаём словарь
    
    return market_dict

# <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
print("ƒ[внутренняя ФУНКЦИЯ] Сохраняем в CSV Новое расписание по методу обработки отдельно каждого символа.") # <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
def process_sessions_conditions(table_name, dfs, symbols_df, directory_data_temp_files, directory_data_log_files):
    print("ƒ[внутренняя ФУНКЦИЯ] Сохраняем в CSV Новое расписание по методу обработки отдельно каждого символа.")
    def time_to_minutes(t):                         # Функция для конвертации времени в минуты
        if t is None:
            return None
        hours, minutes = map(int, t.split(":"))
        return hours * 60 + minutes

    def duplicate_sessions_with_type(df):
        df['type'] = 1
        df_duplicates = df.copy()                                       # 1. Создаём копию для дубликатов
        df_duplicates['type'] = 0                                       # меняем type на 0
        df_combined = pd.concat([df, df_duplicates], ignore_index=True) # 2. Конкатенируем оригинал и дубликаты
        df_combined['_dup'] = [0,1]* (len(df))# 3. Сортируем так, чтобы дубликат шёл сразу после оригинала # создаём вспомогательный индекс: оригинал 0, дубликат 1
        df_combined = df_combined.sort_values(['symbolId','day','_dup']).drop(columns='_dup').reset_index(drop=True)
        df = df_combined        # 4. Обновляем исходный df
        return df

    cols_map = {"name":"name_s"}
    df, symbols_not_found_in_symbols_df, symbols_not_used_in_sessions = imported['merge_left_with_check'](
        df_left= dfs, df_right= symbols_df, left_on= "symbolId", right_on= "symbolId", cols_map=cols_map)

    for col in ["open", "close"]: df[col] = df[col].apply(time_to_minutes)  # Применяем ко всем колонкам open и close перевод времени в минуты с начала суток
    df = duplicate_sessions_with_type(df)
    imported["save_data_log_work_file"](df, table_name+"_sessions.csv", directory_data_temp_files, directory_data_log_files)

    return df

ƒ [ внутренние ФУНКЦИИ ] используемые в этом файле:
ƒ[внутренняя ФУНКЦИЯ] Сортировка колонок в нужном порядке.
ƒ[внутренняя ФУНКЦИЯ] Проверка словарей на одинаковые ключи, значения и пары key:value.
ƒ[внутренняя ФУНКЦИЯ] Фильтрация DataFrame по суффиксам.
ƒ[внутренняя ФУНКЦИЯ] Создание словаря {name_market: id} с проверкой на один ключ — несколько значений.
ƒ[внутренняя ФУНКЦИЯ] Сохраняем в CSV Новое расписание по методу обработки отдельно каждого символа.


In [3]:
# Подключение к SQL серверу и загрузка исходных данных ББ<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''

#dot_big_sql_main_cred= "own_platform\\credits\\own_platforn_sql_main_main_01.txt"           # Получаем Данные с ПРОД сервера
dot_big_sql_main_cred= "own_platform\\credits\\own_platforn_sql_main_stage_01.txt"
print(f"Путь к файлу с данными для подключения к SQL серверу: [ {dot_big_sql_main_cred} ]")

# загрузка данных SQL таблиц для дальнейшей работы <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
symbols_sessions_df = imported["get_sql_tab"]   ("SELECT * FROM `symbolsSessions`",    dot_big_sql_main_cred) 
symbols_df          = imported["get_sql_tab"]   ("SELECT * FROM `symbols`",            dot_big_sql_main_cred)
symbols_markets_df  = imported["get_sql_tab"]   ("SELECT * FROM `symbolsMarkets`",     dot_big_sql_main_cred)

Путь к файлу с данными для подключения к SQL серверу: [ own_platform\credits\own_platforn_sql_main_stage_01.txt ]
получаем данные SELECT * FROM `symbolsSessions`; из: 10.1.0.8 pma.y.d main_stage_01
получаем данные SELECT * FROM `symbols`; из: 10.1.0.8 pma.y.d main_stage_01
получаем данные SELECT * FROM `symbolsMarkets`; из: 10.1.0.8 pma.y.d main_stage_01


In [4]:
print("Преобразование времени и формирование интервалов (поддержка нескольких сессий)...")

# --- ЧАСТЬ 1: ПОДГОТОВКА ВРЕМЕНИ И ИНТЕРВАЛОВ ---
print("Преобразование времени и хронологическая сортировка сессий...")

df_prep = symbols_sessions_df.copy()

# 1. Сначала сортируем весь датафрейм, чтобы сессии внутри дня шли по порядку
# Сортируем по символу, типу, дню и ВРЕМЕНИ открытия
df_prep = df_prep.sort_values(by=['symbolId', 'type', 'day', 'open'])

# 2. Преобразуем минуты в ЧЧ:ММ
def mins_to_hm(x):
    if pd.isna(x): return ""
    return f"{int(x)//60:02d}:{int(x)%60:02d}"

df_prep['open_hm'] = df_prep['open'].apply(mins_to_hm)
df_prep['close_hm'] = df_prep['close'].apply(mins_to_hm)

df_prep['interval'] = df_prep['open_hm'] + "-" + df_prep['close_hm']    # 3. Создаем строку интервала

# 4. Группируем. Благодаря sort_values выше, интервалы склеятся хронологически # Результат будет всегда: "00:00-19:59<br>21:00-23:59"
df_grouped = df_prep.groupby(['symbolId', 'day', 'type'])['interval'].apply(
    lambda x: '<br>'.join(x)
).reset_index()

# --- ЧАСТЬ 2: ТРАНСПОНИРОВАНИЕ (PIVOT) ---
print("Транспонирование (Pivot) для получения дней в колонках...")

# Разворачиваем данные
df_sessions_transposed = df_grouped.pivot(
    index='symbolId', 
    columns=['type', 'day'], 
    values='interval'
)

# Сплющиваем MultiIndex ДО того, как сделаем reset_index()
# Теперь мы точно знаем, что здесь только пары (type, day)
new_cols = []
for col in df_sessions_transposed.columns:
    stype, day = col
    new_cols.append(f"{stype}_{day}")

df_sessions_transposed.columns = new_cols

# Теперь возвращаем symbolId из индекса в колонки
df_sessions_transposed = df_sessions_transposed.reset_index()
# --- ЧАСТЬ 3: ОБОГАЩЕНИЕ ДАННЫМИ (БЕЗ ИЗМЕНЕНИЙ) ---
print("Обогащение данными о рынках и тикерах...")

cols_map_sym = {"name":"name_s", "displayName":"displayName_s", "marketId":"marketId_s", "tradeMode":"tradeMode_s"}
df_sessions_enriched, symbols_not_found, _ = imported['merge_left_with_check'](
    df_left=df_sessions_transposed, df_right=symbols_df, left_on="symbolId", right_on="symbolId", cols_map=cols_map_sym)

cols_map_mkt = {"name": "name_market"}   
df_sessions_enriched, _, _ = imported['merge_left_with_check'](
    df_left=df_sessions_enriched, df_right=symbols_markets_df, left_on="marketId_s", right_on="id", cols_map=cols_map_mkt)

# --- ЧАСТЬ 4: ПЕРЕМЕЩЕНИЕ КОЛОНОК И СОХРАНЕНИЕ ---
df = df_sessions_enriched.copy()
df = df[df['tradeMode_s'] == 4].copy()                                                      # Оставляем только активные символы
list_col_name = ['name_s', 'displayName_s', 'marketId_s', 'name_market','tradeMode_s']
df_sessions_enriched = imported["move_column"](df, list_col_name, False, 0, 1).copy()

print('\n', "Сохранение итоговой таблицы...")
imported["save_data_log_work_file"](df_sessions_enriched, "df_sessions_enriched.csv", directory_data_temp_files, directory_data_log_files)
imported["pd_set_option"]("df_sessions_enriched", df_sessions_enriched, 5)

Преобразование времени и формирование интервалов (поддержка нескольких сессий)...
Преобразование времени и хронологическая сортировка сессий...
Транспонирование (Pivot) для получения дней в колонках...
Обогащение данными о рынках и тикерах...
 
 [dif] Функция добавления новых колонок с пустыми значениями и изменения положения этих колонок  
 list_col_name = ['name_s', 'displayName_s', 'marketId_s', 'name_market', 'tradeMode_s'];  
 new_position = 0, new_position_step = 1

 Сохранение итоговой таблицы...
💾 Полный путь к сохраняемому файлу: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\Reading\working_data_files\df_sessions_enriched.csv
💾 Полный путь к сохраняемому файлу: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\Reading\log_data_files\2026-05-19 12-54-51.df_sessions_enriched.csv

df_sessions_enriched  (1,655 строк × 21 колонок)


,name_s,displayName_s,marketId_s,name_market,tradeMode_s,symbolId,0_0,1_0,0_1,1_1,0_2,1_2,0_3,1_3,0_4,1_4,0_5,1_5,0_6,1_6,id
0,AUDCAD,AUD / CAD,2.0,Minor,4.0,1,21:00-23:59,21:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-20:59,00:00-20:59,NaN,NaN,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1997,AEDUSD,AEDUSD,33.0,CFDs - Stocks United States,4.0,10829,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,33.0


In [5]:

check_cols = ['0_0', '1_0', '0_1', '1_1', '0_2', '1_2', '0_3', '1_3', '0_4', '1_4', '0_5', '1_5', '0_6', '1_6'] # 1. Список колонок
filtered_cols = [col for col in check_cols if not col.startswith('0_')] # 2. Оставляем только те, что НЕ начинаются на "0_" # (То есть удаляем маску "0_*", оставляя только "1_*")

days_map = {'1_0':'ВС','1_1':'ПН','1_2':'ВТ','1_3':'СР','1_4':'ЧТ','1_5':'ПТ','1_6':'СБ'}   # 3. Словарь для переименования (маска "1_X" -> День недели)

cols_to_drop = [col for col in df_sessions_enriched.columns if col.startswith('0_')]        # Сначала удаляем лишние колонки (те, что начинаются на 0_)
df_final = df_sessions_enriched.drop(columns=cols_to_drop)
df_final = df_final.rename(columns=days_map)                                                # Затем переименовываем оставшиеся по словарю

def clean_numeric_to_str(df, target_cols):
    df = df.copy()
    for col in target_cols:
        if col in df.columns:
            # fillna(0) нужен, так как int не дружит с NaN
            # Затем переводим в int (убирает .0) и в str
            df[col] = df[col].fillna(0).astype(int).astype(str)
            # Если 0 нам не нужен (был NaN), можно заменить обратно на пустую строку
            df[col] = df[col].replace('0', '-') 
            
    return df

target_cols = ['marketId_s', 'tradeMode_s', 'id', 'symbolId']    # Список колонок, которые мы точно хотим видеть как целые числа
df_final = clean_numeric_to_str(df_final, target_cols).copy()

df_sessions_enriched_2 = df_final.copy()

print("Оставшиеся колонки:", df_final.columns.tolist())
imported["save_data_log_work_file"](df_sessions_enriched_2, "df_sessions_enriched_2.csv", directory_data_temp_files, directory_data_log_files)
imported["pd_set_option"]("df_sessions_enriched_2", df_sessions_enriched_2, 50) # Вывод ДФ для проверки

Оставшиеся колонки: ['name_s', 'displayName_s', 'marketId_s', 'name_market', 'tradeMode_s', 'symbolId', 'ВС', 'ПН', 'ВТ', 'СР', 'ЧТ', 'ПТ', 'СБ', 'id']
💾 Полный путь к сохраняемому файлу: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\Reading\working_data_files\df_sessions_enriched_2.csv
💾 Полный путь к сохраняемому файлу: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\Reading\log_data_files\2026-05-19 12-54-51.df_sessions_enriched_2.csv

df_sessions_enriched_2  (1,655 строк × 14 колонок)


,name_s,displayName_s,marketId_s,name_market,tradeMode_s,symbolId,ВС,ПН,ВТ,СР,ЧТ,ПТ,СБ,id
0,AUDCAD,AUD / CAD,2,Minor,4,1,21:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-20:59,NaN,2
1,AUDCHF,AUD / CHF,2,Minor,4,2,21:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-20:59,NaN,2
2,AUDCNH,AUD / CNH,77,Exotic,4,3,21:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-20:59,NaN,77
3,AUDDKK,AUD / DKK,77,Exotic,4,4,21:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-20:59,NaN,77
4,AUDHUF,AUD / HUF,77,Exotic,4,7,21:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-20:59,NaN,77
5,AUDINR,AUD / INR,77,Exotic,4,8,21:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-20:59,NaN,77
6,AUDJPY,AUD / JPY,2,Minor,4,9,21:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-20:59,NaN,2
7,AUDMEX,AUD / MEX,77,Exotic,4,10,21:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-20:59,NaN,77
8,AUDNOK,AUD / NOK,77,Exotic,4,11,21:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-20:59,NaN,77
9,AUDNZD,AUD / NZD,2,Minor,4,12,21:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-20:59,NaN,2


In [6]:
# Технический код для проверки <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
df = df_sessions_enriched.copy()
df_filtered = df[df["marketId_s"] == 78]
print("symbols_df[\"marketId\"] == 78: ", len(df_filtered))
symbolId_list = df_filtered["symbolId"].tolist()
print("symbolId_list: ", symbolId_list)
imported["pd_set_option"]("df_filtered", df_filtered, 50)

symbols_df["marketId"] == 78:  13
symbolId_list:  [10484, 10487, 10488, 10493, 10514, 10516, 10517, 10518, 10519, 10534, 10545, 10552, 10677]

df_filtered  (13 строк × 21 колонок)


,name_s,displayName_s,marketId_s,name_market,tradeMode_s,symbolId,0_0,1_0,0_1,1_1,0_2,1_2,0_3,1_3,0_4,1_4,0_5,1_5,0_6,1_6,id
1984,DEYAAR,DEYAAR,78.0,CFDs Stocks Arab Emirates,4.0,10484,NaN,NaN,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,NaN,NaN,78.0
1985,Dubai.Fi'cial.Market,Dubai Fi'cial Market,78.0,CFDs Stocks Arab Emirates,4.0,10487,NaN,NaN,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,NaN,NaN,78.0
1986,Dubai.Investments,Dubai Investments,78.0,CFDs Stocks Arab Emirates,4.0,10488,NaN,NaN,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,NaN,NaN,78.0
1987,Emaar.Development,Emaar Development,78.0,CFDs Stocks Arab Emirates,4.0,10493,NaN,NaN,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,NaN,NaN,78.0
1988,Union.Properties,Union Properties,78.0,CFDs Stocks Arab Emirates,4.0,10514,NaN,NaN,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,NaN,NaN,78.0
1989,ADNOC.Distribution,ADNOC Distribution,78.0,CFDs Stocks Arab Emirates,4.0,10516,NaN,NaN,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,NaN,NaN,78.0
1990,ADNOC.Drilling,ADNOC Drilling,78.0,CFDs Stocks Arab Emirates,4.0,10517,NaN,NaN,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,NaN,NaN,78.0
1991,ADNOC.Gas,ADNOC Gas,78.0,CFDs Stocks Arab Emirates,4.0,10518,NaN,NaN,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,NaN,NaN,78.0
1992,ADNOC.Logistics,ADNOC Logistics,78.0,CFDs Stocks Arab Emirates,4.0,10519,NaN,NaN,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,NaN,NaN,78.0
1993,Aldar.Properties,Aldar Properties,78.0,CFDs Stocks Arab Emirates,4.0,10534,NaN,NaN,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,06:00-11:00,NaN,NaN,78.0


In [7]:
# [ФУНКЦИЯ] Сохранение DataFrame в HTML с полной стилизацией (шрифт, границы, отступы) <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# '''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''

import os
import json

def df_to_html(df_to_save, table_title, name_file):
    # 1. Подготовка данных (как и раньше)
    if 'example_patterns' in df_to_save.columns:
        def pretty_patterns(patterns):
            if not patterns: return "Нет"
            return "<br><br>".join(json.dumps(p, indent=2).replace('\n', '<br>') for p in patterns)
        df_to_save = df_to_save.copy()
        df_to_save['example_patterns'] = df_to_save['example_patterns'].apply(pretty_patterns)

    # 2. Генерируем основной HTML через Styler
    styled = df_to_save.style.hide(axis='index')
    html_body = styled.to_html(escape=False)

    # 3. Формируем ПОЛНЫЙ HTML-файл со встроенными стилями
    # Это гарантирует шрифт и границы независимо от настроек браузера
    full_html = f"""
    <html>
    <head>
    <meta charset="UTF-8">
    <style>
        body {{
            font-family: sans-serif;
            padding: 20px;
        }}
        .title-caption {{
            font-size: 18px;
            font-weight: bold;
            margin-bottom: 10px;
            font-family: sans-serif;
        }}
        table {{
            border-collapse: collapse;
            width: 100%;
            font-family: 'Consolas', 'Monaco', 'Courier New', monospace; /* Моноширинный шрифт */
            font-size: 13px;
        }}
        th {{
            background-color: #e6f0ff;
            border: 1px solid #aaa;
            padding: 8px;
            text-align: center;
            font-weight: bold;
        }}
        td {{
            border: 1px solid #ccc; /* Границы ячеек */
            padding: 6px;
            text-align: left;
            vertical-align: top;
            white-space: pre-wrap;
        }}
        tr:nth-child(even) {{
            background-color: #f9f9f9;
        }}
        tr:hover {{
            background-color: #f1f1f1;
        }}
    </style>
    </head>
    <body>
        <div class="title-caption">{table_title}</div>
        {html_body}
    </body>
    </html>
    """

    # 4. Сохранение
    output_file = f"{name_file}.html"
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(full_html)
    
    print(os.path.abspath(output_file))
    print(f"✅ HTML-файл [ {table_title} ] успешно сохранён с полной стилизацией \n")

In [8]:
# 1. Определяем колонки с расписаниями (тип_день)
# Ищем все колонки, которые заканчиваются на число (день недели 0-6)

#schedule_cols = [c for c in df_sessions_enriched.columns if re.search(r'_\d$', c)]

day_names = ['ВС', 'ПН', 'ВТ', 'СР', 'ЧТ', 'ПТ', 'СБ']          # Определяем список дней недели, которые теперь стали именами колонок
schedule_cols = [c for c in df_final.columns if c in day_names] # Собираем только те колонки из списка выше, которые реально есть в вашем DataFrame

# [ФУНКЦИЯ] Проверка на наличие внутри одного кластера инструментов с разными расписаниями <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
def check_enriched_consistency(df, cluster_col='name_market'):          # Группируем по кластеру и считаем количество уникальных расписаний для каждого дня
    diff_check = df.groupby(cluster_col)[schedule_cols].nunique().max(axis=1)   # nunique() > 1 означает, что в этом кластере у инструментов разные сессии
    problem_clusters = diff_check[diff_check > 1].index.tolist()                # Находим проблемные кластеры
    atoms_df = df[df[cluster_col].isin(problem_clusters)].copy()
    similar_df = df[~df[cluster_col].isin(problem_clusters)].copy()
    return atoms_df, similar_df
# >>> The END [check_enriched_consistency] <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<

# [применяем функцию ]Разделяем данные на "атомарные" (разные) и "похожие" (одинаковые) внутри кластеров рынков
# Включает Проверка на наличие внутри одного кластера инструментов с разными расписаниями
atoms_df, similar_df = check_enriched_consistency(df_sessions_enriched_2, cluster_col='name_market')

if not atoms_df.empty:
    imported["save_data_log_work_file"](atoms_df, "atoms_df.csv", directory_data_temp_files, directory_data_log_files)
    df_to_html(atoms_df, "Инструменты c отличающимися расписаниями внутри Кластера", "sessions_atoms_enriched")   # Формируем HTML отчёты

if not similar_df.empty:
    imported["save_data_log_work_file"](similar_df, "similar_df.csv", directory_data_temp_files, directory_data_log_files)        
    df_to_html(similar_df, "Развёрнутое расписание по инструментам внутри однотипных Кластеров", "sessions_full_enriched") # Отчет 1: Полный список (все инструменты)
    
    cols_to_exclude = ['symbolId', 'name_s', 'displayName_s', 'tradeMode_s', 'id']# Отчет 2: Сводный # Исключаем индивидуальные данные инструментов
    clusters_summary = similar_df.drop(columns=cols_to_exclude, errors='ignore').drop_duplicates()
    
    imported["save_data_log_work_file"](clusters_summary, "clusters_summary.csv", directory_data_temp_files, directory_data_log_files)
    df_to_html(clusters_summary, "Инструменты одинаковыми расписаниями внутри Кластера", "sessions_markets_summary")

💾 Полный путь к сохраняемому файлу: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\Reading\working_data_files\atoms_df.csv
💾 Полный путь к сохраняемому файлу: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\Reading\log_data_files\2026-05-19 12-55-16.atoms_df.csv
c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\Reading\ipynb_files\sessions_atoms_enriched.html
✅ HTML-файл [ Инструменты c отличающимися расписаниями внутри Кластера ] успешно сохранён с полной стилизацией 

💾 Полный путь к сохраняемому файлу: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\Reading\working_data_files\similar_df.csv
💾 Полный путь к сохраняемому файлу: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\Reading\log_data_files\2026-05-19 12-55-21.similar_df.csv
c:\unique_data\rep_fo_metatrader_